In [2]:
import re
import shutil
import pandas as pd
from google.cloud import bigquery
import openpyxl
import os
import glob
from openpyxl.styles import PatternFill, Font
from datetime import datetime

# Helper function to clean illegal characters from a string
def clean_string(value):
    if isinstance(value, str):
        # Remove non-printable characters (including ASCII control characters)
        return re.sub(r'[\x00-\x1F\x7F]', '', value)
    return value

# Function to clean specific columns in the dataframe
def clean_dataframe(df, columns):
    for column in columns:
        if column in df.columns:
            df[column] = df[column].apply(lambda x: str(x).strip() if isinstance(x, str) else x)
    return df


# Helper function to find files with specific prefixes
def find_files(step_prefix, unmatched_prefix):
    """
    Look for files with the given prefixes for Step and unmatched files.
    """
    step_file = glob.glob(f"{step_prefix}*.xlsx")
    unmatched_file = glob.glob(f"{unmatched_prefix}*.csv")

    return step_file[0] if step_file else None, unmatched_file[0] if unmatched_file else None


# Function to get BigQuery keys
def get_bigquery_keys():
    key1 = input("Enter the first 16-character key for BigQuery: ")
    key2 = input("Enter the second 16-character key for BigQuery: ")
    return key1, key2


# Function to fetch backend data from BigQuery
def get_backend_data(key1, key2):
    client = bigquery.Client(project='tealbook-prd')  # Specify your GCP project ID
    query = f"""
    SELECT
      DISTINCT(r.internalSupplierId) AS internalSupplierId,
      ARRAY_TO_STRING(og.domains,"|") AS domains,
      og.name,
      og.supplier.locations[SAFE_OFFSET(0)].address as Primary_Address
    FROM
        `tealbook-prd.tealbook.RelationshipSpendLineItems` r
    LEFT JOIN
      `tealbook-prd.tealbook.OrgUnit` og
    ON
      SUBSTR(r.supplier, 9, 16) = SUBSTR(og.__key__.path, 8,16)
      AND RIGHT(r.supplier, 16) = SUBSTR(og.__key__.path, 37,16)
    WHERE
        ARRAY_TO_STRING(og.domains,"|") != "missing.link"
        AND SUBSTR(r.__key__.path, 8,16) = "{key1}"
        AND SUBSTR(r.__key__.path, 37,16) = "{key2}";
    """
    query_job = client.query(query)
    return query_job.to_dataframe()


# Helper function to find a column by header name (case-insensitive)
def find_column_by_header(sheet, header_name):
    header_name_normalized = header_name.strip().lower()
    for col in range(1, sheet.max_column + 1):
        cell_value = sheet.cell(row=1, column=col).value
        if cell_value and cell_value.strip().lower() == header_name_normalized:
            return col
    raise ValueError(f"Header '{header_name}' not found in the sheet.")


# Function to process step files
def process_step_files(step_file, unmatched_file, title_suffix, backend_data):
    """
    Process a step file and unmatched file to produce Step 1.3 and Step 1.4 files.
    """
    # Adjust Step 1.3 file name: Replace "Step 1.2" with "Step 1.3" and update the suffix
    step_1_3_file = step_file.replace("Step 1.2", "Step 1.3").replace("Client Domains", "Primary Addresses")
    if "Step 1.1" in step_file:
        step_1_3_file = step_file.replace("Step 1.1", "Step 1.3").replace("Client Domains", "Primary Addresses")

    shutil.copy(step_file, step_1_3_file)
    
    # Load the Step 1.3 file
    workbook = openpyxl.load_workbook(step_1_3_file)
    vendor_master = workbook['Vendor Master']

    # Add new columns to the Vendor Master tab
    unmatched_col = vendor_master.max_column + 1
    vendor_master.cell(row=1, column=unmatched_col).value = "Unmatched"

    primary_address_col = vendor_master.max_column + 1
    vendor_master.cell(row=1, column=primary_address_col).value = "Primary Addresses"

    backend_domains_col = vendor_master.max_column + 1
    vendor_master.cell(row=1, column=backend_domains_col).value = "Backend Domains"

    # Define required columns before using them
    required_columns = ['internal_supplier_id']
    
    # Load unmatched and backend data
    unmatched_data = pd.read_csv(unmatched_file)
    for col in required_columns:
        if col not in unmatched_data.columns:
            raise KeyError(f"Expected column '{col}' is missing in the unmatched file.")
    unmatched_ids = unmatched_data['internal_supplier_id'].astype(str).str.strip().tolist()

    backend_df = pd.DataFrame(backend_data)

    # Populate Unmatched column
    for row in range(2, vendor_master.max_row + 1):
        internal_supplier_id = str(vendor_master.cell(row=row, column=1).value).strip()
        if internal_supplier_id in unmatched_ids:
            vendor_master.cell(row=row, column=unmatched_col).value = internal_supplier_id

    # Populate "Primary Addresses" and "Backend Domains"
    for row in range(2, vendor_master.max_row + 1):
        internal_supplier_id = str(vendor_master.cell(row=row, column=1).value).strip()
        match = backend_df[backend_df['internalSupplierId'] == internal_supplier_id]
        if not match.empty:
            vendor_master.cell(row=row, column=primary_address_col).value = clean_string(match['Primary_Address'].values[0])
            vendor_master.cell(row=row, column=backend_domains_col).value = match['domains'].values[0]

    # Replace values in the "web_domain" and "Complete Address" columns
    web_domain_col = find_column_by_header(vendor_master, "web_domain")
    complete_address_col = find_column_by_header(vendor_master, "Complete Address")

    for row in range(2, vendor_master.max_row + 1):
        unmatched_value = vendor_master.cell(row=row, column=unmatched_col).value
        if unmatched_value:
            vendor_master.cell(row=row, column=web_domain_col).value = None
            vendor_master.cell(row=row, column=complete_address_col).value = vendor_master.cell(row=row, column=primary_address_col).value

    # Apply formatting to Step 1.3
    format_worksheet(vendor_master, highlight_col=primary_address_col)
    
    # Save Step 1.3
    workbook.save(step_1_3_file)

    # Print confirmation for Step 1.3
    print(f"Formatting applied to {step_1_3_file}")

    # Define Step 1.4 file name
    step_1_4_file = step_1_3_file.replace("1.3", "1.4").replace("Primary Addresses", "Backend Domains")

    # Create Step 1.4 file
    shutil.copy(step_1_3_file, step_1_4_file)

    # Modify Step 1.4 for Backend Domains
    workbook = openpyxl.load_workbook(step_1_4_file)
    vendor_master = workbook['Vendor Master']
    
    # Read headers and locate relevant columns
    headers = [vendor_master.cell(row=1, column=col).value for col in range(1, vendor_master.max_column + 1)]
    
    # Locate the Unmatched column and rename it to Unmatched 1.2
    if "Unmatched" in headers:
        unmatched_col = headers.index("Unmatched") + 1
        vendor_master.cell(row=1, column=unmatched_col).value = "Unmatched 1.2"
    else:
        raise ValueError("Unmatched column is missing in the sheet.")
    
    # Locate the Backend Domains column
    if "Backend Domains" not in headers:
        backend_domains_col = vendor_master.max_column + 1  # Add new column at the end
        vendor_master.cell(row=1, column=backend_domains_col).value = "Backend Domains"
    else:
        backend_domains_col = headers.index("Backend Domains") + 1
    
    # Clean Backend Domains column by removing pipe delimiters
    for row in range(2, vendor_master.max_row + 1):
        backend_value = vendor_master.cell(row=row, column=backend_domains_col).value
        if backend_value:
            # Remove everything after and including a pipe delimiter, and strip spaces
            backend_value = str(backend_value).split("|")[0].strip()
            vendor_master.cell(row=row, column=backend_domains_col).value = backend_value
    
    # Debug print for verification
    backend_domains_values = [vendor_master.cell(row=row, column=backend_domains_col).value for row in range(2, vendor_master.max_row + 1)]
    print(f"Cleaned Backend Domains values: {backend_domains_values}")
    
    # Formatting adjustments for Step 1.4
    format_worksheet(vendor_master, highlight_col=backend_domains_col)  # Highlight Backend Domains column
    
    # Save Step 1.4
    workbook.save(step_1_4_file)
    print(f"Formatting applied to {step_1_4_file}")
  
def format_worksheet(sheet, highlight_col=None):
    """
    Format the worksheet with specified column highlighting.
    """
    # Define styles
    teal_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    orange_fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")
    bold_font = Font(bold=True)

    # Apply teal highlight and bold font to headers
    for col in range(1, sheet.max_column + 1):
        header_cell = sheet.cell(row=1, column=col)
        header_cell.fill = teal_fill
        header_cell.font = bold_font

    # Remove any existing filters
    if sheet.auto_filter.ref:
        sheet.auto_filter.ref = None

    # Set column widths
    column_widths = [16, 26, 46, 20, 15, 30]  # A-F
    for col_idx, width in enumerate(column_widths, start=1):
        sheet.column_dimensions[openpyxl.utils.get_column_letter(col_idx)].width = width

    # Highlight specified column in orange
    if highlight_col:
        for row in range(2, sheet.max_row + 1):
            cell = sheet.cell(row=row, column=highlight_col)
            if cell.value:  # Highlight non-empty cells
                cell.fill = orange_fill

# Main function
def main():
    # Step 1: Look for Step 1.2 and 1.2 - unmatched files
    step_1_2_file, unmatched_1_2_file = find_files("Step 1.2", "1.2 - unmatched")

    if step_1_2_file and unmatched_1_2_file:
        # Step 2: Run BigQuery query
        key1, key2 = get_bigquery_keys()
        backend_data = get_backend_data(key1, key2)

        # Process files for Step 1.2
        process_step_files(step_1_2_file, unmatched_1_2_file, "Primary Addresses", backend_data)
    else:
        # Step 14: Look for Step 1.1 and 1.1 - unmatched
        step_1_1_file, unmatched_1_1_file = find_files("Step 1.1", "1.1 - unmatched")
        if not step_1_1_file or not unmatched_1_1_file:
            raise FileNotFoundError("MISSING 1.1 FILES")

        # Duplicate Step 1.1 to Step 1.3 and process
        key1, key2 = get_bigquery_keys()
        backend_data = get_backend_data(key1, key2)

        process_step_files(step_1_1_file, unmatched_1_1_file, f"{datetime.today().strftime('%Y-%m-%d')} - Primary Addresses", backend_data)
    
if __name__ == "__main__":
    main()


Enter the first 16-character key for BigQuery:  4811560459436032
Enter the second 16-character key for BigQuery:  5629499534213120


Formatting applied to Step 1.3_accor - SDP Input - 2025-02-10 - Primary Addresses.xlsx
Cleaned Backend Domains values: ['zurichna.com', None, None, 'zuora.com', 'zumex.com', 'zumaandsons.com', None, None, None, None, None, None, None, None, None, None, 'zodiacglazing.com', None, None, None, None, None, 'zingipop.com', None, None, None, 'zephirinsbakery.com', 'zenbooth.net', 'zeldascatering.com', None, 'zeel.com', None, None, None, None, None, None, None, None, None, 'zappworx.com', None, None, None, None, None, None, None, None, None, 'ywcaquebec.qc.ca', 'yvr.ca', None, 'yvonneosei.com', 'hereafter.la', None, None, None, None, None, None, None, None, None, None, 'yuryesli.com', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'yinyuan-writes.com', None, 'ypsilon-group.com', None, 'welcometoyouth.com', None, 'yourspotbbq.com', 'yoursourcing-solutions.com', None, None, 'youniqueproducts.com', None, 'youngsfinewine.com', None, 'yesco.com', No